In [1]:
import ast
import logging
import os
import sqlite3
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger("etl")

DATA_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
DB_PATH = os.path.join(DATA_DIR, "PerfectPetClub.db")
CRM_PATH = os.path.join(DATA_DIR, "PerfectPetClub_CRM_report.csv")
LOG_PATH = os.path.join(DATA_DIR, "PerfectPetClub_server_log.csv")
SURVEY_PATH = os.path.join(DATA_DIR, "PerfectPetClub_survey.csv")
OUT_DIR = os.path.join(DATA_DIR, "output")
PARQUET_PATH = os.path.join(OUT_DIR, "customer_churn.parquet")
QUALITY_PATH = os.path.join(OUT_DIR, "data_quality_report.csv")

os.makedirs(OUT_DIR, exist_ok=True)
log.info("Working directory: %s", DATA_DIR)

INFO: Working directory: C:\Users\JAYA SUDHESH\Downloads\project


In [2]:
def extract():
    """Load all raw data sources into a dict of DataFrames."""
    raw = {}

    
    con = sqlite3.connect(DB_PATH)
    for table in ["customer", "subscription", "subscription_type", "product",
                  "product_category", "sale", "sale_product", "payment",
                  "product_review"]:
        raw[table] = pd.read_sql_query(f'SELECT * FROM "{table}"', con)
    con.close()
    log.info("Loaded %d SQLite tables", len(raw))

    
    raw["crm"] = pd.read_csv(CRM_PATH)
    log.info("CRM report: %d rows", len(raw["crm"]))

    raw["server_log"] = pd.read_csv(
        LOG_PATH, usecols=["datetime", "session_id", "login"])
    log.info("Server log: %d rows", len(raw["server_log"]))

    
    if os.path.exists(SURVEY_PATH):
        raw["survey"] = pd.read_csv(SURVEY_PATH)
        log.info("Survey: %d rows", len(raw["survey"]))
    else:
        log.warning("Survey file not found - survey features will be empty")

    return raw


raw = extract()
{key: df.shape for key, df in raw.items()}

INFO: Loaded 9 SQLite tables


INFO: CRM report: 7585 rows


INFO: Server log: 67249 rows


INFO: Survey: 221 rows


{'customer': (221, 9),
 'subscription': (221, 5),
 'subscription_type': (3, 3),
 'product': (194, 4),
 'product_category': (26, 2),
 'sale': (23188, 6),
 'sale_product': (59759, 4),
 'payment': (23188, 4),
 'product_review': (10990, 5),
 'crm': (7585, 4),
 'server_log': (67249, 3),
 'survey': (221, 5)}

In [3]:
def download_survey():
    """Best-effort download of the survey CSV from the UH block storage."""
    try:
        import boto3
        s3 = boto3.client(
            "s3",
            endpoint_url="http://vuh-la-specs-s3-1.herts.ac.uk:9000",
            aws_access_key_id="gcx63N1ce4HiITy0GQ2y",
            aws_secret_access_key="lK0znVWRwCmaMoqLR4LX2CESqdfSL8VGA4rN2jbV",
        )
        resp = s3.list_objects_v2(Bucket="comsci-student-s3-1")
        keys = [o["Key"] for o in resp.get("Contents", [])
                if "survey" in o["Key"].lower()]
        log.info("Survey objects found: %s", keys)
        for key in keys:
            if key.lower().endswith(".csv"):
                s3.download_file("comsci-student-s3-1", key, SURVEY_PATH)
                log.info("Downloaded %s -> %s", key, SURVEY_PATH)
                return True
    except Exception as exc: 
        log.warning("Survey download failed: %s", exc)
    return False




In [4]:
def transform_customers(raw):
    """Base customer table + churn label + subscription attributes."""
    customers = raw["customer"][["id", "email", "subscription_id"]].rename(
        columns={"id": "customer_id"})
    subs = raw["subscription"].rename(columns={
        "id": "subscription_id",
        "start_date": "subscription_start",
        "end_date": "subscription_end",
    })
    plans = raw["subscription_type"][["id", "name", "delivery_service"]].rename(
        columns={"id": "subscription_type_id", "name": "subscription_plan"})

    df = (customers
          .merge(subs, on="subscription_id")
          .merge(plans, on="subscription_type_id"))

    
    df["churn"] = (df["auto_renew"] == 0).astype(int)
    df = df.drop(columns=["auto_renew", "subscription_id", "subscription_type_id"])
    return df


def transform_sales(raw):
    """Per-customer sales aggregates (RFM-style monetary/frequency features)."""
    sales = raw["sale"].copy()
    sales["order_date"] = pd.to_datetime(sales["order_date"])
    sales["expected_date"] = pd.to_datetime(sales["expected_date"], errors="coerce")
    sales["delivered_date"] = pd.to_datetime(sales["delivered_date"], errors="coerce")

    pay = raw["payment"][["id", "amount"]].rename(columns={"id": "payment_id"})
    sales = sales.merge(pay, on="payment_id", how="left")

    sales["late"] = ((sales["delivered_date"].notna())
                     & (sales["expected_date"].notna())
                     & (sales["delivered_date"] > sales["expected_date"])).astype(int)
    sales["delivered"] = sales["delivered_date"].notna().astype(int)

    per_customer = (sales.groupby("customer_id")
        .agg(order_count=("id", "count"),
             total_spend=("amount", "sum"),
             max_order_value=("amount", "max"),
             first_order_date=("order_date", "min"),
             last_order_date=("order_date", "max"),
             delivered_order_count=("delivered", "sum"),
             late_delivery_count=("late", "sum"))
        .reset_index())
    per_customer["avg_order_value"] = (
        per_customer["total_spend"] / per_customer["order_count"]).round(2)
    per_customer["late_delivery_ratio"] = (
        per_customer["late_delivery_count"]
        / per_customer["delivered_order_count"].clip(lower=1)).round(4)

    
    sp = raw["sale_product"].merge(
        raw["sale"][["id", "customer_id"]], left_on="sale_id", right_on="id")
    sp = sp.merge(raw["product"][["id", "product_category_id"]],
                  left_on="product_id", right_on="id")
    basket = (sp.groupby("customer_id")
        .agg(items_purchased=("quantity", "sum"),
             distinct_products=("product_id", "nunique"),
             distinct_categories=("product_category_id", "nunique"))
        .reset_index())

    return per_customer.merge(basket, on="customer_id", how="left")


def transform_reviews(raw):
    """Per-customer product review aggregates."""
    rev = raw["product_review"]
    return (rev.groupby("customer_id")
        .agg(review_count=("id", "count"),
             avg_star_rating=("star_rating", "mean"),
             negative_review_ratio=("star_rating",
                 lambda s: (s <= 2).mean()))
        .reset_index()
        .round({"avg_star_rating": 3, "negative_review_ratio": 4}))


def transform_crm(raw, snapshot_date):
    """Per-customer customer-service contact features."""
    crm = raw["crm"].copy().reset_index(drop=True)
    crm["contact_row_id"] = crm.index
    crm["recieved_datatime"] = pd.to_datetime(crm["recieved_datatime"])
    crm["is_complaint"] = (crm["type"].str.lower() == "complaint").astype(int)
    crm["is_query"] = (crm["type"].str.lower() == "query").astype(int)

    
    crm["platforms"] = crm["platform"].map(
        lambda x: " ".join(ast.literal_eval(x)) if isinstance(x, str) else "")
    for chan in ["e-mail", "live-chat", "telephone"]:
        crm[f"contact_{chan.replace('-', '_')}"] = (
            crm["platforms"].str.contains(chan, regex=False)).astype(int)

    cutoff_90d = snapshot_date - timedelta(days=90)
    agg = (crm.groupby("customer_id")
        .agg(contact_count=("contact_row_id", "count"),
             complaint_count=("is_complaint", "sum"),
             query_count=("is_query", "sum"),
             email_contact_count=("contact_e_mail", "sum"),
             chat_contact_count=("contact_live_chat", "sum"),
             phone_contact_count=("contact_telephone", "sum"),
             last_contact_date=("recieved_datatime", "max"))
        .reset_index())
    recent = (crm[crm["recieved_datatime"] >= cutoff_90d]
        .groupby("customer_id").size().rename("contacts_last_90d"))
    agg = agg.merge(recent, on="customer_id", how="left")
    return agg


def transform_logins(raw, snapshot_date):
    """Per-customer web-app login/session features, keyed via email."""
    logins = raw["server_log"].copy()
    logins["datetime"] = pd.to_datetime(logins["datetime"])
    logins["login"] = logins["login"].str.lower().str.strip()

    cutoff_30d = snapshot_date - timedelta(days=30)
    agg = (logins.groupby("login")
        .agg(login_count=("datetime", "count"),
             active_days=("datetime", lambda s: s.dt.date.nunique()),
             session_count=("session_id", "nunique"),
             first_login=("datetime", "min"),
             last_login=("datetime", "max"))
        .reset_index())
    recent = (logins[logins["datetime"] >= cutoff_30d]
        .groupby("login").size().rename("logins_last_30d"))
    agg = agg.merge(recent, on="login", how="left")
    return agg


def transform_survey(raw):
    """Survey responses keyed by email; empty frame if file absent."""
    cols = ["email", "reported_income", "nps_recommend",
            "satisfaction_score", "online_comparison_freq"]
    if "survey" not in raw:
        
        return pd.DataFrame({
            "email": pd.Series(dtype="string"),
            "reported_income": pd.Series(dtype="string"),
            "nps_recommend": pd.Series(dtype="float64"),
            "satisfaction_score": pd.Series(dtype="float64"),
            "online_comparison_freq": pd.Series(dtype="float64"),
            "survey_responded": pd.Series(dtype="int64"),
        })
    sv = raw["survey"].copy()
    sv.columns = [c.strip().lower() for c in sv.columns]
    keep = [c for c in cols if c in sv.columns]
    sv = sv[keep].drop_duplicates(subset="email")
    sv = sv.rename(columns={"email": "_email_key"})
    sv["_email_key"] = sv["_email_key"].str.lower().str.strip()
    for c in keep:
        if c != "email" and c != "reported_income":
            sv[c] = pd.to_numeric(sv[c], errors="coerce")
    sv["survey_responded"] = 1
    return sv

In [5]:
def transform(raw):
    """Join all per-customer aggregates and engineer additional features."""
    
    snapshot_date = pd.to_datetime(raw["sale"]["order_date"]).max()
    log.info("Snapshot date: %s", snapshot_date)

    df = transform_customers(raw)
    df["_email_key"] = df["email"].str.lower().str.strip()

    for name, part in [
        ("sales", transform_sales(raw)),
        ("reviews", transform_reviews(raw)),
        ("crm", transform_crm(raw, snapshot_date)),
        ("survey", transform_survey(raw)),
    ]:
        on = "_email_key" if name == "survey" else "customer_id"
        df = df.merge(part, on=on, how="left")
        log.info("Joined %s -> shape %s", name, df.shape)

    
    logins = transform_logins(raw, snapshot_date).rename(columns={"login": "email"})
    email_map = df[["customer_id", "email"]].assign(
        email=df["email"].str.lower().str.strip())
    logins = logins.merge(email_map[["customer_id", "email"]], on="email", how="left")
    logins_agg = (logins.groupby("customer_id")
        .agg(login_count=("login_count", "sum"),
             active_days=("active_days", "max"),
             session_count=("session_count", "sum"),
             first_login=("first_login", "min"),
             last_login=("last_login", "max"),
             logins_last_30d=("logins_last_30d", "sum"))
        .reset_index())
    df = df.merge(logins_agg, on="customer_id", how="left")
    log.info("Joined logins -> shape %s", df.shape)

    
    df["subscription_start"] = pd.to_datetime(df["subscription_start"])
    df["subscription_end"] = pd.to_datetime(df["subscription_end"])
    df["tenure_days"] = (snapshot_date - df["subscription_start"]).dt.days
    df["subscription_length_days"] = (
        df["subscription_end"] - df["subscription_start"]).dt.days
    df["days_since_last_order"] = (
        snapshot_date - pd.to_datetime(df["last_order_date"])).dt.days
    df["order_span_days"] = (
        pd.to_datetime(df["last_order_date"])
        - pd.to_datetime(df["first_order_date"])).dt.days
    df["days_since_last_login"] = (
        snapshot_date - pd.to_datetime(df["last_login"])).dt.days
    df["days_since_last_contact"] = (
        snapshot_date - pd.to_datetime(df["last_contact_date"])).dt.days

    
    df["complaint_ratio"] = (df["complaint_count"]
                             / df["contact_count"].clip(lower=1)).round(4)
    df["items_per_order"] = (df["items_purchased"]
                             / df["order_count"].clip(lower=1)).round(2)
    df["reviews_per_order"] = (df["review_count"]
                               / df["order_count"].clip(lower=1)).round(4)
    df["logins_per_order"] = (df["login_count"]
                              / df["order_count"].clip(lower=1)).round(4)

    
    count_cols = [c for c in df.columns if c.endswith(("_count", "_days"))
                  or c in ("order_count", "contact_count", "login_count",
                           "active_days", "session_count", "items_purchased",
                           "distinct_products", "distinct_categories",
                           "contacts_last_90d", "logins_last_30d",
                           "survey_responded", "email_contact_count",
                           "chat_contact_count", "phone_contact_count",
                           "complaint_count", "query_count")]
    df[count_cols] = df[count_cols].fillna(0)

    
    df["survey_responded"] = df["survey_responded"].fillna(0).astype(int)

    
    drop_cols = ["subscription_start", "subscription_end", "first_order_date",
                 "last_order_date", "first_login", "last_login",
                 "last_contact_date", "email", "_email_key"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return df, snapshot_date


def data_quality_checks(df, snapshot_date):
    """Run a battery of checks and return a tidy report DataFrame."""
    checks = []

    def add(check, column, value, status="PASS"):
        checks.append({"check": check, "column": column,
                       "value": value, "status": status})

    add("row_count", "-", len(df))
    add("unique_customers", "customer_id",
        df["customer_id"].nunique(),
        "PASS" if df["customer_id"].is_unique else "FAIL")
    add("null_customer_id", "customer_id", int(df["customer_id"].isna().sum()),
        "PASS" if df["customer_id"].notna().all() else "FAIL")
    add("churn_is_binary", "churn",
        sorted(df["churn"].dropna().unique().tolist()),
        "PASS" if set(df["churn"].unique()) <= {0, 1} else "FAIL")
    add("duplicate_rows", "-", int(df.duplicated().sum()),
        "PASS" if not df.duplicated().any() else "FAIL")
    add("churn_rate", "churn", round(df["churn"].mean(), 4))

    
    missing = df.isna().mean().sort_values(ascending=False)
    for col, frac in missing[missing > 0].items():
        add("missing_fraction", col, round(float(frac), 4),
            "PASS" if frac < 0.5 else "WARN")

    
    for col, lo, hi, name in [
        ("avg_star_rating", 1, 5, "avg_star_rating_in_1_5"),
        ("nps_recommend", 0, 10, "nps_in_0_10"),
        ("satisfaction_score", 0, 10, "satisfaction_in_0_10"),
        ("online_comparison_freq", 0, 10, "comparison_freq_in_0_10"),
        ("tenure_days", 0, None, "tenure_non_negative"),
        ("days_since_last_order", 0, None, "recency_non_negative"),
    ]:
        if col not in df.columns:
            continue
        s = df[col].dropna()
        if s.empty:
            add(name, col, "all null", "WARN")
            continue
        ok = (s >= lo).all() and (hi is None or (s <= hi).all())
        add(name, col, f"min={s.min()}, max={s.max()}",
            "PASS" if ok else "FAIL")

   
    neg = {c: int((df[c] < 0).sum()) for c in df.columns
           if pd.api.types.is_numeric_dtype(df[c])}
    neg = {k: v for k, v in neg.items() if v}
    add("negative_values", "-", neg or "none",
        "PASS" if not neg else "FAIL")

    return pd.DataFrame(checks)


def load(df, quality_report):
    """Persist the final dataset as Parquet plus the quality report."""
    df.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")
    quality_report.to_csv(QUALITY_PATH, index=False)
    log.info("Wrote %s (%d rows, %d cols)", PARQUET_PATH, *df.shape[::-1])
    return PARQUET_PATH

In [6]:
df, snapshot_date = transform(raw)
quality_report = data_quality_checks(df, snapshot_date)
load(df, quality_report)

print(f"Dataset: {df.shape[0]} customers x {df.shape[1]} columns")
print(f"Churn rate: {df['churn'].mean():.1%}  (1 = cancelled auto-renewal)")
quality_report

INFO: Snapshot date: 2026-11-10 17:26:39.509000


INFO: Joined sales -> shape (221, 20)


INFO: Joined reviews -> shape (221, 23)


INFO: Joined crm -> shape (221, 31)


INFO: Joined survey -> shape (221, 36)


INFO: Joined logins -> shape (221, 42)


C:\Users\JAYA SUDHESH\AppData\Local\Temp\ipykernel_13552\3955306476.py:142: Pandas4Warning: The 'fastparquet' engine is deprecated and will be removed in a future version. Use 'pyarrow' instead.
  df.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")


INFO: Wrote C:\Users\JAYA SUDHESH\Downloads\project\output\customer_churn.parquet (43 rows, 221 cols)


Dataset: 221 customers x 43 columns
Churn rate: 38.9%  (1 = cancelled auto-renewal)


,check,column,value,status
0,row_count,-,221,PASS
1,unique_customers,customer_id,221,PASS
2,null_customer_id,customer_id,0,PASS
3,churn_is_binary,churn,"[0, 1]",PASS
4,duplicate_rows,-,0,PASS
5,churn_rate,churn,0.3891,PASS
6,missing_fraction,days_since_last_contact,0.095,PASS
7,missing_fraction,complaint_ratio,0.095,PASS
8,missing_fraction,negative_review_ratio,0.0362,PASS
9,missing_fraction,avg_star_rating,0.0362,PASS


In [7]:
df.head(10).T

,0,1,2,3,4,5,6,7,8,9
customer_id,1,2,3,4,5,6,7,8,9,10
subscription_plan,SuperSaver,SuperSaver,Premium,Premium,SuperSaver,SuperSaver,SuperSaver,SuperSaver,SuperSaver,Premium
delivery_service,Economy72,Economy72,SignedFor24,SignedFor24,Economy72,Economy72,Economy72,Economy72,Economy72,SignedFor24
churn,1,0,0,0,1,1,1,0,1,0
order_count,20,253,304,1055,44,2,276,289,127,121
total_spend,74455,1036961,1284692,9577504,179738,2497,1177018,1222736,460851,514351
max_order_value,9797,19795,22394,37993,10097,2198,19646,19545,12995,19796
delivered_order_count,19,242,293,1044,40,1,268,278,119,110
late_delivery_count,1,8,8,49,3,0,11,11,6,7
avg_order_value,3722.75,4098.66,4225.96,9078.2,4084.95,1248.5,4264.56,4230.92,3628.75,4250.83
